# slice-view-mutation — worked example 3: Accumulate values into rows via repeated slice writes

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `slice-view-mutation`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

You can loop over rows and accumulate values in-place using slice views: `mat[i, :] += delta[i]` adds delta[i] to every element of row i without creating a new matrix. This pattern appears in scatter operations and custom backward functions where you're distributing gradient contributions back into a buffer.

## Worked solution

**Step 1 — Create a buffer matrix of shape (3, 4).** Initially all zeros.

**Step 2 — Create a delta tensor.** Each row has a different increment to add.

**Step 3 — Loop and accumulate.** `buf[i, :] += delta[i]` writes through the row view. Each iteration modifies a different row of `buf` in place.

**Step 4 — Verify.** Each row of `buf` should equal the corresponding row of `delta` (since we started from zeros).

In [ ]:
import torch as t

t.manual_seed(3)

buf = t.zeros(3, 4)
delta = t.tensor([
    [1.0, 2.0, 3.0, 4.0],
    [5.0, 0.0, -1.0, 2.0],
    [0.0, 7.0, 0.0, -3.0],
])

print('Buffer before:\n', buf)
for i in range(3):
    buf[i, :] += delta[i]

print('Buffer after:\n', buf)
assert t.allclose(buf, delta), 'buffer should equal delta after accumulation'
print('Accumulation correct!')